In [20]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

content_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

print("Setup complete")

Setup complete


# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked action queue

My lane is **content refresh opportunity scoring**.

This playbook turns the validated model output into a ranked list for **human review**. The score is not an automatic instruction to change a page.

The queue uses four practical actions:

- `REVIEW_FOR_REFRESH` — high measured decline risk plus enough existing visibility to make review worthwhile.
- `CHECK_CTR` — the page has visibility but weak observed click capture, so the title, snippet, and search-intent match should be checked first.
- `MONITOR` — there is some measured risk, but the evidence is not strong enough for an immediate content change.
- `HOLD` — there is not enough measured evidence to prioritize the page right now.

Each row also receives one reason code so a human can see why it was ranked:

- `HIGH_RISK_VISIBLE`
- `LOW_CTR_VISIBLE`
- `STALE_VISIBLE`
- `MODERATE_RISK`
- `LOW_EVIDENCE`

The decay/refresh insight is used carefully: older pages with existing visibility may be worth reviewing, but page age alone does not prove that a refresh will improve performance.

In [21]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

# --------------------------------------------------
# Rebuild the same honest feature frame
# --------------------------------------------------

playbook_query = f"""
WITH daily_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS past_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS past_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_sum_position
                ELSE 0
            END
        ) AS past_sum_position,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
              AND gsc_data_available IS TRUE
        ) AS gsc_observed_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
              AND gsc_data_available IS TRUE
        ) AS outcome_days

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

content_dates AS (
    SELECT
        client_hash_id,
        content_hash_id,
        is_published,
        is_deleted,

        NULLIF(
            GREATEST(
                COALESCE(
                    CASE
                        WHEN last_optimized_date <= DATE '2026-03-21'
                        THEN last_optimized_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_updated_date <= DATE '2026-03-21'
                        THEN content_updated_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_created_date <= DATE '2026-03-21'
                        THEN content_created_date
                    END,
                    DATE '1900-01-01'
                )
            ),
            DATE '1900-01-01'
        ) AS last_known_update_date

    FROM read_parquet('{content_path}')
)

SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.past_impressions,
    d.past_clicks,

    ROUND(
        100.0 * d.past_clicks /
        NULLIF(d.past_impressions, 0),
        4
    ) AS past_ctr,

    ROUND(
        1.0 * d.past_sum_position /
        NULLIF(d.past_impressions, 0),
        4
    ) AS past_avg_position,

    d.gsc_observed_days,

    DATE_DIFF(
        'day',
        c.last_known_update_date,
        DATE '2026-03-21'
    ) AS days_since_update,

    CASE
        WHEN
            (1.0 * d.outcome_impressions / d.outcome_days)
            <
            0.80 * (
                1.0 * d.past_impressions /
                d.gsc_observed_days
            )
        THEN 1
        ELSE 0
    END AS future_decline

FROM daily_windows d

INNER JOIN content_dates c
    ON d.client_hash_id = c.client_hash_id
   AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_published IS TRUE
    AND COALESCE(c.is_deleted, FALSE) IS FALSE
    AND d.gsc_observed_days >= 7
    AND d.outcome_days >= 5
    AND d.past_impressions >= 100
"""

playbook_frame = con.sql(playbook_query).df()

# --------------------------------------------------
# Deterministic row order for reproducible splitting
# --------------------------------------------------

playbook_frame = (
    playbook_frame
    .sort_values(
        ["client_hash_id", "content_hash_id"]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# Same five honest Week-5 features
# --------------------------------------------------

features = [
    "past_impressions",
    "past_clicks",
    "past_ctr",
    "past_avg_position",
    "gsc_observed_days",
]

X = playbook_frame[features].astype(float)
y = playbook_frame["future_decline"].astype(int)
groups = playbook_frame["client_hash_id"]


# --------------------------------------------------
# Grouped client split
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups,
    )
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx],
)

model_scores = rf_model.predict_proba(
    X.iloc[test_idx]
)[:, 1]


# --------------------------------------------------
# Build held-out human-review queue
# --------------------------------------------------

queue = playbook_frame.iloc[test_idx].copy()

queue["model_score"] = model_scores


def classify_row(row):

    # High measured risk + useful existing visibility
    if (
        row["model_score"] >= 0.60
        and row["past_impressions"] >= 500
    ):
        return pd.Series([
            "High-risk visible",
            "REVIEW_FOR_REFRESH",
            "HIGH_RISK_VISIBLE",
            4,
        ])

    # Existing visibility, but weak click capture
    if (
        row["past_impressions"] >= 500
        and row["past_ctr"] < 0.20
    ):
        return pd.Series([
            "Visible low-CTR",
            "CHECK_CTR",
            "LOW_CTR_VISIBLE",
            3,
        ])

    # Older page with enough visibility to justify review
    if (
        row["days_since_update"] >= 180
        and row["past_impressions"] >= 500
    ):
        return pd.Series([
            "Stale visible",
            "REVIEW_FOR_REFRESH",
            "STALE_VISIBLE",
            3,
        ])

    # Some measured risk, but not enough for immediate action
    if row["model_score"] >= 0.50:
        return pd.Series([
            "Moderate-risk",
            "MONITOR",
            "MODERATE_RISK",
            2,
        ])

    # Weak evidence
    return pd.Series([
        "Low-evidence",
        "HOLD",
        "LOW_EVIDENCE",
        1,
    ])


queue[
    [
        "archetype",
        "action",
        "reason_code",
        "action_priority",
    ]
] = queue.apply(
    classify_row,
    axis=1,
)


# --------------------------------------------------
# Rank the action queue
# --------------------------------------------------

ranked_queue = (
    queue
    .sort_values(
        by=[
            "action_priority",
            "model_score",
            "past_impressions",
            "client_hash_id",
            "content_hash_id",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "rank",
    range(1, len(ranked_queue) + 1),
)


# --------------------------------------------------
# Reproducibility check
# --------------------------------------------------

train_clients = set(
    playbook_frame.iloc[train_idx]["client_hash_id"]
)

test_clients = set(
    playbook_frame.iloc[test_idx]["client_hash_id"]
)

print("Rows in held-out queue:", len(ranked_queue))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients & test_clients),
)


# --------------------------------------------------
# Show action counts and top 20
# --------------------------------------------------

print("\nAction counts:")
print(
    ranked_queue["action"].value_counts()
)

print("\nArchetype to action mapping:")
display(
    ranked_queue[
        [
            "archetype",
            "action",
            "reason_code",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["action", "archetype"]
    )
    .reset_index(drop=True)
)

display(
    ranked_queue[
        [
            "rank",
            "content_hash_id",
            "model_score",
            "archetype",
            "action",
            "reason_code",
            "past_impressions",
            "past_ctr",
            "past_avg_position",
            "days_since_update",
        ]
    ].head(20)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in held-out queue: 4247
Train clients: 31
Test clients: 8
Client overlap: 0

Action counts:
action
HOLD                  1825
MONITOR               1608
CHECK_CTR              538
REVIEW_FOR_REFRESH     276
Name: count, dtype: int64

Archetype to action mapping:


,archetype,action,reason_code
0,Visible low-CTR,CHECK_CTR,LOW_CTR_VISIBLE
1,Low-evidence,HOLD,LOW_EVIDENCE
2,Moderate-risk,MONITOR,MODERATE_RISK
3,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE
4,Stale visible,REVIEW_FOR_REFRESH,STALE_VISIBLE


,rank,content_hash_id,model_score,archetype,action,reason_code,past_impressions,past_ctr,past_avg_position,days_since_update
0,1,content_66cb7a06190f8b36,0.726694,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,908.0,0.0000,0.3183,26
1,2,content_5e2431e72d2dcf2d,0.710408,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,634.0,0.0000,0.2492,26
2,3,content_598935915045b56f,0.708286,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,986.0,0.0000,0.4381,26
3,4,content_9fec22585bac7270,0.707513,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1023.0,0.0000,0.4731,26
4,5,content_ee18e4561fdebed4,0.702736,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,863.0,0.0000,0.4647,8
5,6,content_6194fb8b862a6c3a,0.702095,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1418.0,0.0000,0.1982,26
6,7,content_d6fbfda8973fb759,0.691087,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,961.0,0.0000,0.2653,51
7,8,content_0ef92dd8ea773a64,0.688867,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1677.0,0.0596,71.2260,226
8,9,content_95548071f90fd296,0.688168,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1062.0,0.0000,0.3484,26
9,10,content_2af42daa393a487b,0.686275,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,505.0,0.0000,0.4277,26


### Ranked queue result

The held-out queue contains 4,247 content items.

The playbook assigns:

- 276 items to `REVIEW_FOR_REFRESH`
- 538 items to `CHECK_CTR`
- 1,608 items to `MONITOR`
- 1,825 items to `HOLD`

The queue is intentionally conservative. Most items are not sent directly to a refresh action because the model score is a prioritization signal, not proof that changing the page will improve performance.

The highest-ranked review candidates combine relatively high measured model risk with existing search visibility. Each recommendation also carries a reason code so a reviewer can understand why the item entered the queue.

The archetype-to-action mapping is:

- `High-risk visible` → `REVIEW_FOR_REFRESH`
- `Stale visible` → `REVIEW_FOR_REFRESH`
- `Visible low-CTR` → `CHECK_CTR`
- `Moderate-risk` → `MONITOR`
- `Low-evidence` → `HOLD`

These are decision-support recommendations for human review, not automatic content changes.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is designed to help a content team decide **which pages to review first**.

It takes the validated ranking signal and turns it into a practical queue with actions and reason codes. A reviewer can use the queue to focus attention on pages where there is measured evidence worth investigating.

The playbook is intended for:

- prioritizing human content review
- identifying pages that may deserve a refresh check
- identifying visible pages with weak click capture
- separating immediate review candidates from pages that should only be monitored
- giving the recommendations section of the research paper a clear, reproducible source

### Limits

The playbook does **not** prove that refreshing a page will improve its traffic.

The model was evaluated on held-out clients and showed directional ranking value, but performance remained modest. The recommendations should therefore be interpreted as decision-support signals rather than guaranteed outcomes.

The analysis also has a population limitation: pages need sufficient GSC observations in the later outcome window to be evaluated, so the results apply to the observed subset rather than every page in the portfolio.

Page age, model score, CTR, and visibility should never be interpreted alone as a complete measure of content quality.

### Decay / refresh insight

The earlier analysis showed that age and staleness can help identify pages worth reviewing, but the relationship was not strong enough to support automatic refresh decisions.

The safe operational reading is:

> Older pages with meaningful existing visibility may be worth reviewing before lower-evidence pages, but a human must confirm that the page is actually outdated or declining for a reason a refresh could address.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every recommendation in this playbook requires a human check before any content change is made.

For `REVIEW_FOR_REFRESH`, the reviewer should check:

- Is the page actually outdated or incomplete?
- Is the observed decline seasonal or temporary?
- Has search intent changed?
- Does another page compete for the same demand?
- Does the page already rank well enough that a smaller change would be safer?
- Is there enough business value to justify the work?

For `CHECK_CTR`, the reviewer should first inspect the title, snippet, search intent, and SERP context. A low CTR does not automatically mean the page content itself needs rewriting.

For `MONITOR`, no immediate content change is required. The page should be watched for another measurement window before taking action.

For `HOLD`, the current evidence is too weak to justify work.

### Cost / value thinking

Review effort should be spent first where there is both measured risk and existing visibility.

A high-risk page with meaningful impressions may justify a deeper review because there is more existing traffic to protect. A low-visibility page with uncertain evidence should usually receive less review time.

This is a prioritization rule, not a revenue claim. The model does not measure the financial return of a refresh.

### What should NOT be automated

The playbook must not automatically:

- rewrite or publish content
- delete or unpublish pages
- change redirects, canonicals, or indexing settings
- change titles or metadata without human review
- treat a model score as proof that a refresh will work
- act on pages with weak or insufficient evidence
- make spending or revenue decisions from the score alone

The final decision stays with a human reviewer.

In [22]:
review_policy = pd.DataFrame(
    [
        {
            "action": "REVIEW_FOR_REFRESH",
            "human_check": "Check freshness, intent, seasonality, overlap, and business value.",
            "automation_allowed": False,
        },
        {
            "action": "CHECK_CTR",
            "human_check": "Inspect title, snippet, SERP context, and intent before changing content.",
            "automation_allowed": False,
        },
        {
            "action": "MONITOR",
            "human_check": "Wait for another measurement window before acting.",
            "automation_allowed": False,
        },
        {
            "action": "HOLD",
            "human_check": "No action unless new evidence appears.",
            "automation_allowed": False,
        },
    ]
)

print("Actions requiring human review:", len(review_policy))
print(
    "Any action allowed to automate:",
    review_policy["automation_allowed"].any()
)

review_policy

Actions requiring human review: 4
Any action allowed to automate: False


,action,human_check,automation_allowed
0,REVIEW_FOR_REFRESH,"Check freshness, intent, seasonality, overlap,...",False
1,CHECK_CTR,"Inspect title, snippet, SERP context, and inte...",False
2,MONITOR,Wait for another measurement window before act...,False
3,HOLD,No action unless new evidence appears.,False


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

This playbook is not a production monitoring system. The goal is only to define a few practical checks that would tell me when the model or queue should be reviewed again.

I would re-check the model when:

- precision@10 or precision@20 drops noticeably from the validated result
- the positive base rate changes a lot
- the distribution of important inputs such as impressions, CTR, or observed GSC days shifts
- new clients are added with very different traffic patterns
- GSC availability changes enough to alter the usable population
- the model has not been reviewed for a full content cycle

A retrain should happen only after checking whether the change comes from real drift, new data coverage, or a problem in the pipeline.

The goal is not to retrain constantly. The goal is to notice when the old validation result may no longer describe the current data.

In [23]:
monitoring_triggers = pd.DataFrame(
    [
        {
            "trigger": "Precision drop",
            "check": "Precision@10 or precision@20 falls materially below the validated result.",
            "response": "Review errors first; retrain only if the drop is persistent.",
        },
        {
            "trigger": "Base-rate shift",
            "check": "The share of observed future declines changes meaningfully.",
            "response": "Recalculate evaluation metrics and review threshold usefulness.",
        },
        {
            "trigger": "Feature drift",
            "check": "Impressions, CTR, position, or observed-day distributions shift.",
            "response": "Compare old vs current distributions before retraining.",
        },
        {
            "trigger": "New client mix",
            "check": "New clients have traffic patterns different from the validation population.",
            "response": "Run grouped validation again on the updated client set.",
        },
        {
            "trigger": "Data availability change",
            "check": "GSC availability or usable-history depth changes materially.",
            "response": "Re-check population filters and feature coverage.",
        },
        {
            "trigger": "Scheduled review",
            "check": "The model has not been reviewed for one full content cycle.",
            "response": "Re-run validation and decide whether retraining is justified.",
        },
    ]
)

print("Monitoring triggers defined:", len(monitoring_triggers))
monitoring_triggers

Monitoring triggers defined: 6


,trigger,check,response
0,Precision drop,Precision@10 or precision@20 falls materially ...,Review errors first; retrain only if the drop ...
1,Base-rate shift,The share of observed future declines changes ...,Recalculate evaluation metrics and review thre...
2,Feature drift,"Impressions, CTR, position, or observed-day di...",Compare old vs current distributions before re...
3,New client mix,New clients have traffic patterns different fr...,Run grouped validation again on the updated cl...
4,Data availability change,GSC availability or usable-history depth chang...,Re-check population filters and feature coverage.
5,Scheduled review,The model has not been reviewed for one full c...,Re-run validation and decide whether retrainin...


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [24]:
import os
import json

os.makedirs("work/outputs", exist_ok=True)

# --------------------------------------------------
# Queue export for the paper
# --------------------------------------------------

queue_export = ranked_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "model_score",
        "archetype",
        "action",
        "reason_code",
        "past_impressions",
        "past_clicks",
        "past_ctr",
        "past_avg_position",
        "gsc_observed_days",
        "days_since_update",
    ]
].copy()

queue_path = "work/outputs/action_playbook_ranked_queue.csv"

queue_export.to_csv(
    queue_path,
    index=False,
)


# --------------------------------------------------
# Metrics receipt
# --------------------------------------------------

metrics_receipt = {
    "held_out_rows": int(len(ranked_queue)),
    "review_for_refresh": int(
        (ranked_queue["action"] == "REVIEW_FOR_REFRESH").sum()
    ),
    "check_ctr": int(
        (ranked_queue["action"] == "CHECK_CTR").sum()
    ),
    "monitor": int(
        (ranked_queue["action"] == "MONITOR").sum()
    ),
    "hold": int(
        (ranked_queue["action"] == "HOLD").sum()
    ),
    "random_state": RANDOM_STATE,
    "feature_window": "2026-03-01 to 2026-03-21",
    "decision_moment": "2026-03-21",
    "note": (
        "Decision-support queue for human review. "
        "Actions are not automated recommendations."
    ),
}

metrics_path = "work/outputs/action_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(
        metrics_receipt,
        f,
        indent=2,
    )


# --------------------------------------------------
# Verify exports
# --------------------------------------------------

print("Queue CSV written:", queue_path)
print("Metrics JSON written:", metrics_path)

print("\nQueue rows exported:", len(queue_export))
print("CSV exists:", os.path.exists(queue_path))
print("JSON exists:", os.path.exists(metrics_path))

print("\nMetrics receipt:")
print(json.dumps(metrics_receipt, indent=2))

queue_export.head(10)

Queue CSV written: work/outputs/action_playbook_ranked_queue.csv
Metrics JSON written: work/outputs/action_playbook_metrics.json

Queue rows exported: 4247
CSV exists: True
JSON exists: True

Metrics receipt:
{
  "held_out_rows": 4247,
  "review_for_refresh": 276,
  "check_ctr": 538,
  "monitor": 1608,
  "hold": 1825,
  "random_state": 42,
  "feature_window": "2026-03-01 to 2026-03-21",
  "decision_moment": "2026-03-21",
  "note": "Decision-support queue for human review. Actions are not automated recommendations."
}


,rank,client_hash_id,content_hash_id,model_score,archetype,action,reason_code,past_impressions,past_clicks,past_ctr,past_avg_position,gsc_observed_days,days_since_update
0,1,client_a80fca3f171ed1de,content_66cb7a06190f8b36,0.726694,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,908.0,0.0,0.0000,0.3183,17,26
1,2,client_a80fca3f171ed1de,content_5e2431e72d2dcf2d,0.710408,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,634.0,0.0,0.0000,0.2492,17,26
2,3,client_a80fca3f171ed1de,content_598935915045b56f,0.708286,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,986.0,0.0,0.0000,0.4381,17,26
3,4,client_a80fca3f171ed1de,content_9fec22585bac7270,0.707513,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1023.0,0.0,0.0000,0.4731,18,26
4,5,client_a80fca3f171ed1de,content_ee18e4561fdebed4,0.702736,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,863.0,0.0,0.0000,0.4647,18,8
5,6,client_a80fca3f171ed1de,content_6194fb8b862a6c3a,0.702095,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1418.0,0.0,0.0000,0.1982,15,26
6,7,client_1a730cb2640a1abf,content_d6fbfda8973fb759,0.691087,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,961.0,0.0,0.0000,0.2653,21,51
7,8,client_f623b01661d4bfe4,content_0ef92dd8ea773a64,0.688867,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1677.0,1.0,0.0596,71.2260,21,226
8,9,client_a80fca3f171ed1de,content_95548071f90fd296,0.688168,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,1062.0,0.0,0.0000,0.3484,21,26
9,10,client_a80fca3f171ed1de,content_2af42daa393a487b,0.686275,High-risk visible,REVIEW_FOR_REFRESH,HIGH_RISK_VISIBLE,505.0,0.0,0.0000,0.4277,17,26


### Exports for the paper

This notebook should leave behind the exact files the research paper can reuse next week.

The main export is the ranked action queue. The CSV stays out of git by design, but the notebook regenerates it on every run.

I will also save a small metrics JSON as a receipt so the paper can trace the queue back to this notebook run.

The exported queue contains only pseudonymized IDs, model scores, action labels, reason codes, and supporting measured fields. It does not include private client names, URLs, or future-label columns.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.